In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
df6 = pd.read_csv(r".\log\step2.csv")
df6 = df6.drop(['Unnamed: 0'], axis=1, errors='ignore')
df6

,frequency,355.26607692224155,227.52736891479316,228.1048056598934,508.4579820682637,624.7739429956808,509.2412520609261,746.6584998199332,738.8342325716347,744.5957020753282,...,854.7166023522125,858.2383815524672,851.444148653565,887.258212048309,912.2418726871764,911.7194528327119,1047.003667471031,1047.5534428923963,1000.9177192045671,Max_Oxygen_LifeTime
0,0.2,15.693691,9.443991,1.249940,0.000000,34.094721,0.000000,6.857589,167.325163,5.486071,...,18.577831,0.000000,0.000000,0.000000,386.246099,0.000000,15.371201,12.332707,5.183312,0.459737
1,0.2,0.000000,14.682523,0.000000,0.000000,103.661344,0.000000,31.635450,318.990789,0.000000,...,87.348993,0.000000,43.410868,0.000000,756.836451,1.259714,29.981204,34.012291,0.000000,0.971616
2,0.2,6.163256,29.393991,0.000000,46.937403,237.872811,100.458785,12.768803,501.175525,0.000000,...,58.736495,0.000000,120.452377,0.000000,1054.932155,15.863641,0.000000,12.812941,0.000000,0.911778
3,0.2,0.000000,13.837251,53.544146,0.000000,189.468595,114.004574,47.529882,645.434196,0.000000,...,120.985155,109.912853,163.654026,0.000000,1481.539146,25.550453,54.197931,33.293015,0.000000,0.939168
4,0.2,0.000000,22.551184,0.000000,8.134488,158.228915,106.273152,34.180944,740.674757,0.000000,...,67.573096,0.000000,180.896071,0.000000,1739.831124,12.438134,41.837360,36.183663,11.307395,0.841546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574,1.0,11.021610,0.000000,0.000000,0.000000,15.287786,98.902616,107.145214,952.543208,191.080931,...,0.000000,0.000000,348.778338,68.356739,2267.961964,36.573251,23.226117,12.085988,2.627389,0.738645
575,1.0,16.868346,0.000000,0.000000,39.158921,0.000000,189.268120,92.778647,948.403947,120.379463,...,62.517511,112.731044,259.713704,122.042163,2514.359653,26.381320,0.000000,18.796691,15.718870,0.683652
576,1.0,21.019440,0.000000,0.000000,0.000000,0.000000,40.558919,0.000000,948.886917,49.319891,...,0.000000,0.000000,248.336071,81.273623,2804.664412,32.490893,38.805378,32.950128,15.154763,0.652843
577,1.0,32.316066,0.000000,0.000000,0.000000,0.000000,98.823214,144.473708,1428.856588,38.345568,...,0.000000,158.804880,436.519754,0.000000,3477.884529,56.334457,24.070177,22.405750,7.553939,0.622883


In [ ]:

import numpy as np

import pandas as pd

import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, RepeatedKFold, GridSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

RANDOM_STATE = 4

# -----------------------------
# 0. Data Preparation
# -----------------------------

# It is assumed that df6 has already been defined

X = df6.drop('Max_Oxygen_LifeTime', axis=1).values

y = df6['Max_Oxygen_LifeTime'].values.ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=RANDOM_STATE
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

scaler_full = StandardScaler()

X_scaled_full = scaler_full.fit_transform(X)

# For the full search (you can reduce the number of folds to make it faster)

cv_search = RepeatedKFold(n_splits=3, n_repeats=2, random_state=RANDOM_STATE)   # 3×2 = 6 fold (faster)

cv_eval = RepeatedKFold(n_splits=5, n_repeats=20, random_state=RANDOM_STATE)    # Honest final evaluation

# -----------------------------
# Evaluation Function (exactly the same as the original code)
# -----------------------------

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, X_full_scaled, y_full):

    model.fit(X_tr, y_tr)

    y_tr_pred = model.predict(X_tr)

    y_te_pred = model.predict(X_te)

    rmse_train = np.sqrt(mean_squared_error(y_tr, y_tr_pred))

    rmse_test = np.sqrt(mean_squared_error(y_te, y_te_pred))

    r2_train = r2_score(y_tr, y_tr_pred)

    r2_test = r2_score(y_te, y_te_pred)

    cv_scores = cross_val_score(
        model, X_full_scaled, y_full, cv=cv_eval, scoring='r2', n_jobs=-1
    )

    print(f"\n===== {name} =====")

    print(f"RMSE Train: {rmse_train:.4f} | RMSE Test: {rmse_test:.4f}")

    print(f"R² Train:   {r2_train:.4f} | R² Test:   {r2_test:.4f}")

    print(f"CV R² (mean ± std, {cv_eval.get_n_splits()} folds): "
          f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    return {

        "model": name, "rmse_train": rmse_train, "rmse_test": rmse_test,

        "r2_train": r2_train, "r2_test": r2_test,

        "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),

    }

# -----------------------------
# 1. Define the Full Search Space (all combinations)
# -----------------------------
xgb_param_grid = {

    'n_estimators': [200, 300],

    'learning_rate': [0.05, 0.1],

    'max_depth': [3, 5],

    'subsample': [0.8, 0.9],

    'colsample_bytree': [0.8, 0.9],

    'reg_alpha': [0.01, 0.1],

    'reg_lambda': [0.5, 1.0],

    'min_child_weight': [0.2, 0.4],

}

# -----------------------------
# 2. Full Search with GridSearchCV (all combinations)
# -----------------------------

print("Starting full GridSearchCV search...")

print(f"Total number of combinations: {np.prod([len(v) for v in xgb_param_grid.values()])}")

print(f"Number of validation folds: {cv_search.get_n_splits()}")

print(f"Total number of fits: {np.prod([len(v) for v in xgb_param_grid.values()]) * cv_search.get_n_splits()}")

xgb_grid_search = GridSearchCV(

    XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),

    param_grid=xgb_param_grid,

    cv=cv_search,

    scoring='r2',

    n_jobs=-1,           # Use all CPU cores

    verbose=2,           # Display progress (optional)

)

xgb_grid_search.fit(X_train_scaled, y_train)

print("\nBest parameters found:")

print(xgb_grid_search.best_params_)

print(f"Best R² score (on validation): {xgb_grid_search.best_score_:.4f}")

# -----------------------------
# 3. Evaluate the Final Model with the Best Parameters
# -----------------------------

result = evaluate_model(

    "XGBoost (GridSearchCV - Full Search)",

    xgb_grid_search.best_estimator_,

    X_train_scaled, X_test_scaled, y_train, y_test, X_scaled_full, y

)

# (Optional) Print the final result

print("\nFinal result:")

print(result)

import os

# -----------------------------
# 4. Save the Model and Scaler in the XGBOOSTGridSearchCV Folder
# -----------------------------

# Create the folder (if it does not exist)

os.makedirs("XGBOOSTGridSearchCV", exist_ok=True)

best_model = xgb_grid_search.best_estimator_

joblib.dump(best_model, "XGBOOSTGridSearchCV/xgboost_best_model.pkl")

joblib.dump(scaler, "XGBOOSTGridSearchCV/scaler.pkl")   # Scaler for the training data

print("\nModel and scaler were saved in the 'XGBOOSTGridSearchCV' folder:")

print(" - XGBOOSTGridSearchCV/xgboost_best_model.pkl")

print(" - XGBOOSTGridSearchCV/scaler.pkl")



Starting full GridSearchCV search...
Total number of combinations: 256
Number of validation folds: 6
Total number of fits: 1536
Fitting 6 folds for each of 256 candidates, totalling 1536 fits

Best parameters found:
{'colsample_bytree': 0.9, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 0.2, 'n_estimators': 200, 'reg_alpha': 0.1, 'reg_lambda': 0.5, 'subsample': 0.9}
Best R² score (on validation): 0.3231

===== XGBoost (GridSearchCV - Full Search) =====
RMSE Train: 0.0182 | RMSE Test: 0.0481
R² Train:   0.9646 | R² Test:   0.7412
CV R² (mean ± std, 100 folds): 0.3889 ± 0.0852

Final result:
{'model': 'XGBoost (GridSearchCV - Full Search)', 'rmse_train': np.float64(0.018217055719201753), 'rmse_test': np.float64(0.04813057991594253), 'r2_train': 0.9646417084890045, 'r2_test': 0.7411820426104758, 'cv_mean': np.float64(0.38885364327355454), 'cv_std': np.float64(0.08521940821090508)}

Model and scaler were saved in the 'XGBOOSTGridSearchCV' folder:
 - XGBOOSTGridSearchCV/xgboost

In [ ]:
import pandas as pd
import numpy as np

# Load from the folder

model = joblib.load("XGBOOSTGridSearchCV/xgboost_best_model.pkl")

scaler = joblib.load("XGBOOSTGridSearchCV/scaler.pkl")

# New dataset

df_new = df6.copy()

X_new = df_new.values  # or df_new.drop('Target', axis=1).values

X_new_scaled = scaler.transform(X_new)

preds = model.predict(X_new_scaled)

df_new['Predicted'] = preds

df_new.to_csv('new_predictions.csv', index=False)

print("Predictions have been saved.")

